Visualisation adapted with AI assistance (ChatGPT)

In [2]:
import torch
import torch.nn as nn
import math

In [3]:
class network(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 32),
            nn.Tanh(),
            nn.Linear(32, 3)
        )
        
    def forward(self, x):
        return self.net(x)

In [4]:
model = network()
model.load_state_dict(torch.load("navier_stokes_pinn.pth"))
model.eval()

network(
  (net): Sequential(
    (0): Linear(in_features=3, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): Tanh()
    (6): Linear(in_features=32, out_features=3, bias=True)
  )
)

In [5]:
import matplotlib.pyplot as plt
import imageio
import torch
import os
import imageio


frame_dir = "frames"
os.makedirs(frame_dir, exist_ok=True)

N = 80
x = torch.linspace(-1,1,N)
y = torch.linspace(-1,1,N)

X,Y = torch.meshgrid(x,y,indexing="ij")
grid = torch.stack([X.reshape(-1),Y.reshape(-1)],dim=1)

frames = []

t = 0
dt = 0.1
threshold = 0.009

while True:

    tcol = torch.full((grid.shape[0],1), t)
    pts = torch.cat([grid, tcol], dim=1)

    with torch.no_grad():
        pred = model(pts)

    u = pred[:,0].reshape(N,N).T
    v = pred[:,1].reshape(N,N).T

    speed = torch.sqrt(u**2 + v**2)
    max_vel = speed.max().item()

    print(f"t={t:.2f}  max velocity={max_vel}")

    if max_vel < threshold:
        break

    plt.figure()

    plt.streamplot(
        x.numpy(), y.numpy(),
        u.numpy(), v.numpy(),
        color=speed.numpy(),
        density=1.5
    )

    plt.title(f"t = {t:.2f}")

    fname = os.path.join(frame_dir, f"frame_{len(frames):04d}.png")
    plt.savefig(fname)
    plt.close()

    frames.append(fname)

    t += dt
    
images = [imageio.imread(f) for f in frames]
imageio.mimsave("navier_stokes_decay.gif", images, fps=10)

t=0.00  max velocity=0.9966252446174622
t=0.10  max velocity=0.5805407166481018
t=0.20  max velocity=0.5788204669952393
t=0.30  max velocity=0.5607976913452148
t=0.40  max velocity=0.5393811464309692
t=0.50  max velocity=0.5162422060966492
t=0.60  max velocity=0.494015634059906
t=0.70  max velocity=0.47304102778434753
t=0.80  max velocity=0.4533233046531677
t=0.90  max velocity=0.43511635065078735
t=1.00  max velocity=0.41797924041748047
t=1.10  max velocity=0.4016247093677521
t=1.20  max velocity=0.3859085738658905
t=1.30  max velocity=0.3707207143306732
t=1.40  max velocity=0.35610294342041016
t=1.50  max velocity=0.34199631214141846
t=1.60  max velocity=0.3282560706138611
t=1.70  max velocity=0.31488287448883057
t=1.80  max velocity=0.30188772082328796
t=1.90  max velocity=0.28936606645584106
t=2.00  max velocity=0.27724823355674744
t=2.10  max velocity=0.2655407786369324
t=2.20  max velocity=0.25435349345207214
t=2.30  max velocity=0.24359679222106934
t=2.40  max velocity=0.2332611

C:\Users\basit\AppData\Local\Temp\ipykernel_22184\2151772085.py:62: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images = [imageio.imread(f) for f in frames]
